In [1]:
import zipfile
from multiprocessing import cpu_count
import splitfolders
from tensorflow import keras
import numpy as np
from tqdm import tqdm
import shutil
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import regularizers
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras import layers, models
from tensorflow.keras.models import load_model


In [2]:
print("TensorFlow version:", tf.__version__)
print("CPU cores:", tf.config.threading.get_intra_op_parallelism_threads())
print(f"CPU cores detected: {cpu_count()}")

TensorFlow version: 2.20.0
CPU cores: 0
CPU cores detected: 12


In [5]:
zip_path = "bovine_breeds.zip"


with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall()

In [ ]:
data_dir = 'BovineBreeds_Augmented'
for cls in os.listdir(data_dir):
    path = os.path.join(data_dir, cls)
    print(f"{cls}: {len(os.listdir(path))} images")

In [8]:
input_folder = "BovineBreeds_Augmented"
output_folder = "split_data"

splitfolders.ratio(input_folder,
                   output=output_folder,
                   seed=42,
                   ratio=(0.7, 0.2, 0.1))

Copying files: 7250 files [00:39, 183.44 files/s]


In [13]:
def remove_class(path):
    class_to_remove = '.ipynb_checkpoints'
    folder_path = os.path.join(path, class_to_remove)

    if os.path.exists(folder_path):
        shutil.rmtree(folder_path)
        print(f"{class_to_remove} removed successfully")
    else:
        print("Class not found")

remove_class('split_data/train')
remove_class('split_data/val')
remove_class('split_data/test')

.ipynb_checkpoints removed successfully
.ipynb_checkpoints removed successfully
.ipynb_checkpoints removed successfully


In [3]:
train_dir = 'split_data/train'

class_names_from_dir = sorted(os.listdir(train_dir))
print(class_names_from_dir)

['Alambadi', 'Amritmahal', 'Ayrshire', 'Banni', 'Bargur', 'Bhadawari', 'Brown_Swiss', 'Dangi', 'Deoni', 'Gir', 'Holstein_Friesian', 'Jaffrabadi', 'Jersey', 'Kangayam', 'Kasargod', 'Kenkatha', 'Kherigarh', 'Krishna_Valley', 'Mehsana', 'Murrah', 'Nagori', 'Nili_Ravi', 'Nimari', 'Ongole', 'Red_Sindhi', 'Sahiwal', 'Surti', 'Tharparkar', 'Umblachery']


In [4]:
IMG_SIZE = (300, 300)


train_datagen = ImageDataGenerator(preprocessing_function=preprocess_input,  # ✅ EfficientNet-specific
    rotation_range=15,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    horizontal_flip=True,
)


val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input,
)

train_generator = train_datagen.flow_from_directory(
    'split_data/train',
    target_size=IMG_SIZE,
    batch_size=32,
    class_mode='categorical'
)


val_generator = val_datagen.flow_from_directory(
    'split_data/val',
    target_size=IMG_SIZE,
    batch_size=32,
    class_mode='categorical',

)



test_generator = val_datagen.flow_from_directory(
    'split_data/test',
    target_size=IMG_SIZE,
    batch_size=32,
    class_mode='categorical'

)

Found 5062 images belonging to 29 classes.
Found 1439 images belonging to 29 classes.
Found 749 images belonging to 29 classes.


In [ ]:
base_model = EfficientNetB3(weights='imagenet', include_top=False, input_shape=(300, 300, 3))

base_model.trainable = False

x = GlobalAveragePooling2D()(base_model.output)
x1 = Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001))(x)
x2 = layers.Dropout(0.2)(x1)
x3 = Dense(64, activation = 'relu', kernel_regularizer= regularizers.l2(0.001))(x2)
x4 = layers.Dropout(0.2)(x3)
x5 = Dense(29, activation='softmax')(x4)
model = Model(inputs=base_model.input, outputs=x5)


model.compile(optimizer='Adam', loss='categorical_crossentropy', metrics=['accuracy'])

model.summary()

In [ ]:
# Custom callback to monitor CPU during training
import tensorflow.keras.callbacks as callbacks




checkpoint = ModelCheckpoint(
    filepath="best_model.keras",
    monitor="val_accuracy",        # metric to monitor
    mode="max",                    # because we want maximum accuracy
    save_best_only=True,           # saves only when model improves
    verbose=1
)

early_stop = EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True)



history = model.fit(train_generator, validation_data=val_generator,
    epochs=20,
    callbacks=[early_stop, checkpoint],     
    verbose = 1

)

In [7]:
model_path = "best_model.keras"

loaded_model = load_model(model_path)

loss, accuracy = loaded_model.evaluate(test_generator)
print(loss)
print(accuracy)

24/24 ━━━━━━━━━━━━━━━━━━━━ 38s 1s/step - accuracy: 0.8117 - loss: 0.9868
0.986804187297821
0.8117489814758301
